# Week 3, day 4 (morning) — Worksheet 05: Source-to-target mapping, and staging

> *ETL design starts with source-to-target mapping.* — L03, slide 33

The model is decided. Now it has to be built, and slide 33 says the first
artefact is not code — it is a **mapping**, answering six questions:

- Which source tables feed each target table?
- Which source columns become target columns?
- What transformations or business rules are needed?
- Which tables should be loaded first?
- Which records must be summarized to match the target grain?
- What checks confirm the load is correct?

This worksheet writes that mapping, then does **Step 1 (stage source data)** and
**Step 2 (join and filter records)** from slides 34 and 35. Steps 3 to 7 are
worksheets 06 to 10.

The interesting part is not the joining. It is the two audits in questions 7 and
8 — which source columns nothing uses, and which target columns have no source at
all.

**Question 10 is supposed to raise an error.**

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — Source-to-target mapping and staging. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

SOURCE_TABLES = ["category", "city", "cohort", "course", "discount_type",
                 "employee", "employee_type", "enrollment", "payment_type",
                 "program", "students", "transaction"]

# The target model, exactly as slide 29 draws it.
TARGET = {
    "dim_program":  ["program_id", "source_program_id", "program_name",
                     "program_category", "is_active"],
    "dim_course":   ["course_id", "source_course_id", "course_name",
                     "credit_hours", "is_active"],
    "dim_cohort":   ["cohort_id", "source_cohort_id", "cohort_name",
                     "start_date", "end_date"],
    "dim_student":  ["student_id", "source_student_id", "student_name",
                     "date_of_birth", "city", "state", "country", "is_active"],
    "dim_date":     ["date_id", "full_date", "day_of_week", "day", "week",
                     "month", "quarter", "year", "is_weekend"],
    "dim_promotion": ["promotion_id", "source_discount_type_id",
                      "promotion_name", "discount_amount"],
    "fact_enrollment": ["enrollment_id", "program_id", "course_id", "cohort_id",
                        "student_id", "enrollment_date_id", "promotion_id",
                        "enrollment_count", "tuition_amount", "discount_amount",
                        "net_tuition_amount", "amount_paid_to_date",
                        "is_paid_in_full"],
}

print("source tables:", len(SOURCE_TABLES))
print("target tables:", len(TARGET), "->", ", ".join(TARGET))

PART A — the mapping

### Question 1

Write the source-to-target mapping for `dim_course` in slide 33's format: target column, source table/column, and the ETL logic. Print it as a table, then verify each named source column actually exists.

In [ ]:
############################
## Your Code Here
############################

### Question 2

Slide 34 says staging tables *keep data close to the original source structure* and *preserve source keys, timestamps, and source metadata*. Build `stg_enrollment` that way — every source column unchanged, plus `_loaded_at` and `_source_file`. Print its shape and columns.
> **NOTE:** resist the urge to clean anything here. Staging is a faithful copy; transformation is Step 3.

In [ ]:
############################
## Your Code Here
############################

### Question 3

Slide 34 lists which source tables to stage and why. Build that table for this case study: for each of the six target tables, print the source tables it needs.

In [ ]:
############################
## Your Code Here
############################

### Question 4

Slide 38 says *"load dimensions before facts so fact rows can reference them"*. Derive the load order from question 3's dependencies and print it as numbered waves.

In [ ]:
############################
## Your Code Here
############################

PART B — Step 2: join and filter

### Question 5

Build the working dataset slide 35 describes: start from `stg_enrollment` and join course, program, cohort and student context. Print the row count after each join and confirm the grain never changes.
> **NOTE:** every one of these must be `how="left"` and must not add rows. Check both.

In [ ]:
############################
## Your Code Here
############################

### Question 6

Slide 35 says *"filtering out test, duplicate, cancelled, or inactive records when required"*. Count each candidate for exclusion separately, and the total rows affected once overlaps are accounted for.
> **NOTE:** count them as sets, not by adding the four numbers up. A row can be more than one kind of problem.

In [ ]:
############################
## Your Code Here
############################

PART C — auditing the mapping

### Question 7

Mapping completeness, first direction. Across the twelve source tables, count the total columns and list the ones no target column uses.
> **NOTE:** an unused column is a question, not a defect. Some are genuinely irrelevant; some mean a requirement was missed.

In [ ]:
############################
## Your Code Here
############################

### Question 8

Mapping completeness, the other direction. For every column in `TARGET`, classify it as `direct` (copied), `derived` (computed), or `generated` (a surrogate key). Print the counts.
> **NOTE:** this is the column count that tells you how much work the ELT actually is.

In [ ]:
############################
## Your Code Here
############################

### Question 9

Slide 34 says staging *"makes extraction results traceable"*. Show what `source_course_id` buys: pick course 214, and print its source row and the target `dim_course` row that would be built from it.
> **NOTE:** the surrogate key is new; the source key is what lets you get back to where a row came from.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Finally, break slide 38's rule. Build `fact_enrollment` **before** `dim_promotion` exists by looking up promotion keys in an empty dimension: `dim_promotion.set_index("source_discount_type_id").loc[needed]`. **This is supposed to fail.**

In [ ]:
############################
## Your Code Here
############################